In [48]:
# Adjust to the correct user path
import os
os.chdir('/Users/led7283981/Downloads/mimic-iii-clinical-database-1.4')

# Import required libraries
import pandas as pd

# Load MIMIC-III data tables into DataFrames
# Parse date/time columns for proper datetime types
patients = pd.read_csv('PATIENTS.csv', parse_dates=['DOB'])
admissions = pd.read_csv('ADMISSIONS.csv', parse_dates=['ADMITTIME', 'DISCHTIME', 'DEATHTIME'])
icustays = pd.read_csv('ICUSTAYS.csv', parse_dates=['INTIME', 'OUTTIME'])

# Merge ICUSTAYS with ADMISSIONS on subject and hospital admission IDs
df = pd.merge(icustays, admissions, on=['SUBJECT_ID', 'HADM_ID'], how='inner')
# Merge the PATIENTS table to add demographic info (DOB, gender)
df = pd.merge(df, patients[['SUBJECT_ID', 'DOB', 'GENDER']], on='SUBJECT_ID', how='inner')

# The resulting DataFrame df now contains one row per ICU stay, 
# with columns from all three tables (ICU stay info, admission info, patient info).
print("Merged data shape:", df.shape)
print("Columns:", df.columns.tolist()[:15], "...")  # print first 15 column names as example
df.head(3)  # peek at first 3 rows of the merged DataFrame


Merged data shape: (61532, 31)
Columns: ['ROW_ID_x', 'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'DBSOURCE', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'FIRST_WARDID', 'LAST_WARDID', 'INTIME', 'OUTTIME', 'LOS', 'ROW_ID_y', 'ADMITTIME', 'DISCHTIME'] ...


,ROW_ID_x,SUBJECT_ID,HADM_ID,ICUSTAY_ID,DBSOURCE,FIRST_CAREUNIT,LAST_CAREUNIT,FIRST_WARDID,LAST_WARDID,INTIME,...,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA,DOB,GENDER
0,365,268,110404,280836,carevue,MICU,MICU,52,52,2198-02-14 23:27:38,...,CATHOLIC,SEPARATED,HISPANIC OR LATINO,2198-02-11 09:41:00,2198-02-11 15:18:00,DYSPNEA,1,1,2132-02-21,F
1,366,269,106296,206613,carevue,MICU,MICU,52,52,2170-11-05 11:05:29,...,UNOBTAINABLE,SINGLE,WHITE,2170-11-05 07:22:00,2170-11-05 12:15:00,SEPSIS;PILONIDAL ABSCESS,0,1,2130-09-30,M
2,367,270,188028,220345,carevue,CCU,CCU,57,57,2128-06-24 15:05:20,...,JEHOVAH'S WITNESS,MARRIED,UNKNOWN/NOT SPECIFIED,NaN,NaN,CAROTID STENOSIS\CAROTID ANGIOGRAM AND STENT,0,1,2048-05-26,M


In [49]:
import pandas as pd
from datetime import date

# ─── 0. (Assume df is already loaded with ICU stays and has DOB, INTIME, OUTTIME, ETHNICITY, etc.)

# ─── 1. Clean and compute AGE safely ────────────────────────────────────────
# a) Drop rows with missing or clearly invalid DOB/INTIME
df = df[df['DOB'].notna() & df['INTIME'].notna()].copy()
df = df[df['DOB'] < df['INTIME']]                                    # DOB must precede ICU in
df = df[df['DOB'] >= pd.Timestamp('1900-01-01')]                      # optional: drop <1900

# b) Compute age in years via Python dates (won’t overflow)
def compute_age(dob_ts, intime_ts):
    d0 = dob_ts.date()
    d1 = intime_ts.date()
    return (d1 - d0).days / 365.2425

df['AGE'] = df.apply(lambda r: compute_age(r['DOB'], r['INTIME']), axis=1)

# c) Cap at 90 per MIMIC convention
df.loc[df['AGE'] > 89, 'AGE'] = 90

# d) (Optional) keep only adults
df = df[df['AGE'] >= 16].copy()


# ─── 2. Simplify ethnicity ─────────────────────────────────────────────────
def simplify_ethnicity(eth):
    eth = str(eth).upper()
    if 'WHITE'    in eth: return 'WHITE'
    elif 'BLACK'  in eth: return 'BLACK'
    elif 'ASIAN'  in eth: return 'ASIAN'
    elif 'HISPANIC' in eth: return 'HISPANIC'
    else:             return 'OTHER'

df['ETHNICITY_GROUPED'] = df['ETHNICITY'].apply(simplify_ethnicity)


# ─── 3. Ensure GENDER is ready (M/F) ───────────────────────────────────────
# We’ll just leave df['GENDER'] as is, since it’s already 'M' or 'F'


# ─── 4. Cast categorical features ─────────────────────────────────────────
cat_features = [
    'GENDER',
    'ADMISSION_TYPE',
    'INSURANCE',
    'ETHNICITY_GROUPED',
    'FIRST_CAREUNIT'
]
for col in cat_features:
    if col in df.columns:
        df[col] = df[col].astype('category')


# ─── 5. Drop anomalous stays ───────────────────────────────────────────────
# a) Negative ICU length‐of‐stay
mask_neg = (df['OUTTIME'] < df['INTIME'])
print("Dropping rows with negative ICU time difference:", mask_neg.sum())
df = df[~mask_neg].copy()

# b) Exclude stays ending in in‐hospital death (if DEATHTIME exists)
if 'DEATHTIME' in df.columns:
    before = len(df)
    df = df[df['DEATHTIME'].isna()].copy()
    print(f"Dropped {before - len(df)} stays where patient died during ICU stay.")


# ─── 6. Drop raw/unneeded columns ──────────────────────────────────────────
unused_cols = [
    'SUBJECT_ID','HADM_ID','ICUSTAY_ID',
    'INTIME','OUTTIME','ADMITTIME','DISCHTIME',
    'DEATHTIME','DOB','ETHNICITY'
]
to_drop = [c for c in unused_cols if c in df.columns]
df.drop(columns=to_drop, inplace=True)

print("Final feature columns:", df.columns.tolist())



Dropping rows with negative ICU time difference: 0
Dropped 5964 stays where patient died during ICU stay.
Final feature columns: ['ROW_ID_x', 'DBSOURCE', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'FIRST_WARDID', 'LAST_WARDID', 'LOS', 'ROW_ID_y', 'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'DISCHARGE_LOCATION', 'INSURANCE', 'LANGUAGE', 'RELIGION', 'MARITAL_STATUS', 'EDREGTIME', 'EDOUTTIME', 'DIAGNOSIS', 'HOSPITAL_EXPIRE_FLAG', 'HAS_CHARTEVENTS_DATA', 'GENDER', 'AGE', 'ETHNICITY_GROUPED']


In [52]:
import pandas as pd

# ─── 1. Compute ICU_LOS_DAYS ──────────────────────────────────────────
if 'ICU_LOS_DAYS' not in df.columns:
    if {'INTIME','OUTTIME'}.issubset(df.columns):
        # use INTIME/OUTTIME if still present
        df['INTIME']  = pd.to_datetime(df['INTIME'])
        df['OUTTIME'] = pd.to_datetime(df['OUTTIME'])
        df['ICU_LOS_DAYS'] = (
            df['OUTTIME'] - df['INTIME']
        ).dt.total_seconds() / 86400.0
    elif 'LOS' in df.columns:
        # fall back to the LOS column if INTIME/OUTTIME were dropped earlier
        df['ICU_LOS_DAYS'] = df['LOS'].astype(float)
    else:
        raise KeyError(
            "Cannot compute ICU_LOS_DAYS – need either INTIME/OUTTIME or LOS in df.columns"
        )

# ─── 2. Drop raw columns so they don’t leak into the model ────────────
to_drop = [c for c in ['INTIME','OUTTIME','LOS'] if c in df.columns]
df.drop(columns=to_drop, inplace=True)

# ─── 3. Filter out invalid targets ────────────────────────────────────
df = df[df['ICU_LOS_DAYS'].ge(0) & df['ICU_LOS_DAYS'].notna()].copy()

# ─── 4. Quick summary ────────────────────────────────────────────────
print("ICU LOS (days) –",
      "min =", df['ICU_LOS_DAYS'].min(),
      "max =", df['ICU_LOS_DAYS'].max(),
      "median =", df['ICU_LOS_DAYS'].median()
)


ICU LOS (days) – min = 0.0001 max = 173.0725 median = 2.08865


In [53]:
# 1. Handle any remaining missing values in features (if any).
# For simplicity, if any exist, we'll drop those rows.
initial_rows = df.shape[0]
df.dropna(subset=['AGE', 'GENDER', 'ADMISSION_TYPE', 'INSURANCE', 'ETHNICITY_GROUPED', 'FIRST_CAREUNIT', 'ICU_LOS_DAYS'], inplace=True)
print(f"Dropped {initial_rows - df.shape[0]} rows due to missing values in required columns.")

# 2. One-hot encode categorical features
categorical_cols = ['GENDER', 'ADMISSION_TYPE', 'INSURANCE', 'ETHNICITY_GROUPED', 'FIRST_CAREUNIT']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)
print("Data shape after one-hot encoding:", df_encoded.shape)
print("Example encoded columns:", [c for c in df_encoded.columns if 'ADMISSION_TYPE_' in c][:3], "...")

# 3. Separate the feature matrix X and target vector y
X = df_encoded.drop('ICU_LOS_DAYS', axis=1)
y = df_encoded['ICU_LOS_DAYS']

# 4. Split into training and test sets (e.g., 80% train, 20% test)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training samples:", X_train.shape[0], "Test samples:", X_test.shape[0])

# 5. Feature scaling: standardize the numeric columns (here, 'AGE')
from sklearn.preprocessing import StandardScaler
numeric_features = ['AGE']
scaler = StandardScaler()
# Fit on training data only, then transform both train and test
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])


Dropped 0 rows due to missing values in required columns.
Data shape after one-hot encoding: (44774, 38)
Example encoded columns: ['ADMISSION_TYPE_ELECTIVE', 'ADMISSION_TYPE_EMERGENCY', 'ADMISSION_TYPE_URGENT'] ...
Training samples: 35819 Test samples: 8955


In [54]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# ─── assume you already have df with your features + target:
# df['ICU_LOS_DAYS'] = …  # computed above
# X = df.drop("ICU_LOS_DAYS", axis=1)
# y = df["ICU_LOS_DAYS"]

# Split (if you haven’t already)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ─── 1) One‐hot encode all object/category columns
X_train_enc = pd.get_dummies(
    X_train,
    columns=X_train.select_dtypes(include=['object','category']).columns,
    drop_first=True,
    dummy_na=False
)
X_test_enc = pd.get_dummies(
    X_test,
    columns=X_test.select_dtypes(include=['object','category']).columns,
    drop_first=True,
    dummy_na=False
)

# Make sure train/test have same columns
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

# ─── 2) Fit the forest
rf_model = RandomForestRegressor(
    n_estimators=30,      # fewer trees
    max_depth=10,         # shallower trees
    max_features="sqrt",  # fewer features considered per split
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train_enc, y_train)

# ─── 3) Feature importances
importances = pd.Series(
    rf_model.feature_importances_,
    index=X_train_enc.columns
).sort_values(ascending=False)

print("Top 5 features by importance:")
print(importances.head(5))



Top 5 features by importance:
DISCHARGE_LOCATION_HOME                        0.056550
DISCHARGE_LOCATION_REHAB/DISTINCT PART HOSP    0.055933
DISCHARGE_LOCATION_LONG TERM CARE HOSPITAL     0.054013
EDREGTIME_2149-04-29 22:32:00                  0.032033
DIAGNOSIS_VOCAL CORD DISFUNCTION               0.023102
dtype: float64


In [55]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ─── Use the one-hot-encoded test set, not the raw one ────────────────────
y_pred = rf_model.predict(X_test_enc)

# ─── Compute your metrics ──────────────────────────────────────────────────
mae  = mean_absolute_error(y_test,     y_pred)
rmse = np.sqrt(        mean_squared_error(y_test, y_pred))
r2   = r2_score(       y_test,     y_pred)

print(f"Test MAE:  {mae:.2f} days")
print(f"Test RMSE: {rmse:.2f} days")
print(f"Test R²:   {r2:.3f}")


Test MAE:  3.16 days
Test RMSE: 6.21 days
Test R²:   0.021
